# 04 — Aprendizaje supervisado (ocurrencia de incendios)

## [ESP] Objetivo

Ranking de riesgo de ocurrencia a partir de **condiciones cotidianas**
(no satelitales). Positivos = detecciones FIRMS
(`data/processed/expanded/firms_spain_final_lags_index.csv`); negativos =
días de fondo enriquecidos (`background_days.csv`, clave `background_id`).
Métrica principal **PR-AUC**; secundaria ROC-AUC. **No** se producen
probabilidades calibradas: la prevalencia la fija el muestreo de negativos,
no la naturaleza (ver `planning/plan_supervised_implementation.md`, Plan D).

## [ENG] Objective

Risk ranking of fire occurrence from everyday (non-satellite) conditions.
Positives = FIRMS detections; negatives = enriched background days. Primary
metric PR-AUC; no calibrated probabilities.

## Reglas (recordatorio)

- Validación **bloqueada** (leave-one-group-out) por temporada y CCAA: nada de
  K-fold aleatorio (fuga entre días/lugares cercanos).
- Semilla fija `SEED = 42`; unidades crudas; **ninguna columna de satélite**
  entra en `X` (`occurrence_features` las descarta con aviso).
- El one-hot y el escalador se ajustan **solo en el pliegue de entrenamiento**
  y se alinean entre pliegues (categorías no vistas → todo-cero).
- Dependencia: el fondo enriquecido lo construye el equipo. Mientras no exista,
  esta celda 1 fallará en `load_background_days()`.

In [ ]:
# Imports del cuaderno (celda unica). Todas las dependencias aqui; las demas
# celdas no importan nada.
from IPython.display import display
import matplotlib
import matplotlib.pyplot as plt
import geopandas
import sklearn
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.metrics import PrecisionRecallDisplay
from wildfire.analysis.ablation import ablation_verdict
from wildfire.analysis.deployment import ccaa_day_fire_rates
from wildfire.analysis.occurrence_model import (
    categorical_columns,
    categorical_to_dummies,
    continuous_columns,
    fold_scaler_apply,
    fold_scaler_fit,
    model_candidates,
)
from wildfire.data.io import load_csv, save_csv
from wildfire.geo.regions import REGION_NAME_MAP, load_boundaries
from wildfire.processing.occurrence import (
    BACKGROUND_ID_COL,
    EVERYDAY_FEATURES,
    LABEL_COL,
    SEASON_COL,
    add_season,
    assemble_modeling_frame,
    blocked_splits,
    load_background_days,
    occurrence_features,
    occurrence_settings,
    rank_metrics,
    sample_background,
    validate_background_days,
)
from wildfire.processing.reintegration import fit_background_scorer

# Rutas (relativas a PROJECT_ROOT; load_csv/save_csv las anclan) y semilla.
POSITIVES_CSV = "data/processed/expanded/firms_spain_final_lags_index.csv"
PREDICTIONS_CSV = "data/processed/predictions/occurrence_scores.csv"
FOLD_METRICS_CSV = "data/processed/predictions/occurrence_fold_metrics.csv"
SEED = 42
CANDIDATES = ["cluster_meteo", "score_anomalia"]

settings = occurrence_settings()
RATIO = settings["background_ratio"]
LEAKAGE_KM = settings["leakage_max_km"]
PROTOCOLS = tuple(settings["cv_groups"])
print(f"ratio={RATIO} | leakage_max_km={LEAKAGE_KM} | protocolos={PROTOCOLS} | seed={SEED}")

## 1 — Carga, validación y ensamblado

Carga positivos y fondo enriquecido, valida el contrato del fondo (clave
única, esquema cotidiano, sin satélite, sin fugas a ≤ `leakage_max_km` de una
detección del mismo día), muestrea negativos a `ratio` por positivo y ensambla
la matriz de modelado. `season` y `record_id` se re-adjuntan por posición
(`assemble_modeling_frame` solo conserva `EVERYDAY_FEATURES` + `label`).

_[ENG] Load positives + enriched background; validate the background contract (unique key, everyday schema, no satellite, no same-day leakage); sample negatives at the configured ratio and assemble the modeling frame._

In [ ]:
positives = load_csv(POSITIVES_CSV)
if "detection_id" not in positives.columns:
    raise KeyError("Los positivos necesitan la columna detection_id")

background = load_background_days()
report = validate_background_days(background, positives, leakage_max_km=LEAKAGE_KM)
print(
    f"Fondo validado: {report['n_rows']:,} filas | "
    f"duplicados={report['n_duplicate_ids']} | fugas={report['n_leakage']}"
)
if report["nan_report"]:
    display(pd.Series(report["nan_report"], name="NaN").rename_axis("columna").to_frame())
else:
    print("Sin NaN en el fondo.")

sample = sample_background(background, n_positives=len(positives), ratio=RATIO, seed=SEED)
requested = RATIO * len(positives)
note = "" if len(sample) == requested else " (pool insuficiente: se usa todo)"
print(
    f"Positivos: {len(positives):,} | fondo: {len(background):,} | "
    f"pedido ratio {RATIO} = {requested:,} | muestreado: {len(sample):,}{note}"
)

pos_season = add_season(positives)[SEASON_COL].to_numpy()
neg_season = add_season(sample)[SEASON_COL].to_numpy()

# occurrence_features descarta columnas de satelite/sub-diarias con aviso.
pos_features = occurrence_features(positives)
neg_features = occurrence_features(sample)
frame = assemble_modeling_frame(pos_features, neg_features)
frame[SEASON_COL] = np.concatenate([pos_season, neg_season])
frame["record_id"] = np.concatenate(
    [
        positives["detection_id"].astype(str).to_numpy(),
        sample[BACKGROUND_ID_COL].astype(str).to_numpy(),
    ]
)

print("Balance de etiquetas:")
display(frame[LABEL_COL].value_counts().rename("n").rename_axis("label").to_frame())
print("Etiquetas por CCAA:")
display(frame.groupby("ccaa")[LABEL_COL].value_counts().unstack(fill_value=0))
print("Etiquetas por temporada:")
display(frame.groupby(SEASON_COL)[LABEL_COL].value_counts().unstack(fill_value=0))

## 2 — Línea base bloqueada

`fold_eval` ajusta, por pliegue (leave-one-group-out sobre season/ccaa), el
one-hot y el escalador **solo con entrenamiento**, entrena el modelo y calcula
PR-AUC/ROC-AUC en el grupo retenido. La **PR-AUC de línea base = prevalencia**
del pliegue (un clasificador sin habilidad obtiene exactamente eso). Se
descartan pliegues de una sola clase.

_[ENG] Blocked leave-one-group-out baseline: per-fold one-hot/scaler fit on the train fold only; the PR-AUC baseline equals fold prevalence; single-class folds are skipped._

In [ ]:
def prepare_fold(train, test, feature_cols, categorical_cols):
    """One-hot (ajustado en train) + imputacion de medianas + escalado por pliegue."""
    X_train, dummies = categorical_to_dummies(train[feature_cols], categorical_cols)
    X_test, _ = categorical_to_dummies(
        test[feature_cols], categorical_cols, train_columns=dummies
    )
    cont = continuous_columns(X_train)
    medians = X_train[cont].median()
    X_train[cont] = X_train[cont].fillna(medians)
    X_test[cont] = X_test[cont].fillna(medians)
    scaler = fold_scaler_fit(X_train, cont)
    return fold_scaler_apply(X_train, scaler, cont), fold_scaler_apply(X_test, scaler, cont)


def fold_eval(data, feature_cols, categorical_cols, model, protocols=PROTOCOLS):
    """PR-AUC/ROC-AUC por pliegue para un modelo (leave-one-group-out)."""
    rows = []
    for protocol in protocols:
        for split in blocked_splits(data, protocol):
            train, test = data.iloc[split.train_index], data.iloc[split.test_index]
            if train[LABEL_COL].nunique() < 2 or test[LABEL_COL].nunique() < 2:
                print(f"[skip] {protocol}/{split.held_out}: menos de dos clases")
                continue
            X_train, X_test = prepare_fold(train, test, feature_cols, categorical_cols)
            fitted = clone(model).fit(X_train, train[LABEL_COL])
            scores = fitted.predict_proba(X_test)[:, 1]
            rows.append(
                {
                    "protocol": protocol,
                    "held_out": split.held_out,
                    "n_train": len(train),
                    "n_test": len(test),
                    "prevalence": float(test[LABEL_COL].mean()),
                    **rank_metrics(test[LABEL_COL], scores),
                }
            )
    return pd.DataFrame(rows)


floor_name = "logistic"
floor_model = model_candidates(seed=SEED)[floor_name]
baseline_metrics = fold_eval(frame, EVERYDAY_FEATURES, categorical_columns(), floor_model)
display(baseline_metrics)
display(baseline_metrics.groupby("protocol")[["pr_auc", "roc_auc"]].agg(["mean", "min"]))
print(f"PR-AUC sin habilidad (prevalencia media): {baseline_metrics['prevalence'].mean():.3f}")

## 3 — Comparación de modelos candidatos

Los tres modelos provisionales (`LogisticRegression`, `RandomForestClassifier`,
`HistGradientBoostingClassifier`) bajo los mismos pliegues. Media/mínimo de
PR-AUC y ROC-AUC por modelo y protocolo. El mejor por PR-AUC media se usa en
la ablación.

_[ENG] The three provisional models under identical folds; compare mean/min PR-AUC and ROC-AUC; the best by mean PR-AUC feeds the ablation._

In [ ]:
comparisons = []
for name, model in model_candidates(seed=SEED).items():
    table = fold_eval(frame, EVERYDAY_FEATURES, categorical_columns(), model)
    table["model"] = name
    comparisons.append(table)
candidate_metrics = pd.concat(comparisons, ignore_index=True)
comparison_summary = (
    candidate_metrics.groupby("model")[["pr_auc", "roc_auc"]]
    .agg(["mean", "min"])
    .sort_values(("pr_auc", "mean"), ascending=False)
)
display(comparison_summary)
best_name = comparison_summary.index[0]
print(f"Mejor modelo por PR-AUC media: {best_name}")

## 4 — Reintegración (block 03 → 04)

`fit_background_scorer(positives)` ajusta **solo con positivos** el PCA
meteorológico, la tipología k12 y los detectores de anomalías; `.score` etiqueta
(`cluster_meteo`) y puntúa (`score_anomalia`) ambas clases con esos mismos
objetos (nunca reajusta). Se reconstruye la matriz con los candidatos
(`pos_candidates`/`neg_candidates`).

_[ENG] Fit the background scorer on positives only and score both classes; rebuild the frame with the cluster_meteo/score_anomalia candidates._

In [ ]:
scorer = fit_background_scorer(positives, seed=SEED)
pos_candidates = scorer.score(positives)[CANDIDATES]
neg_candidates = scorer.score(sample)[CANDIDATES]

frame_candidates = assemble_modeling_frame(
    pos_features,
    neg_features,
    pos_candidates=pos_candidates,
    neg_candidates=neg_candidates,
)
frame_candidates[SEASON_COL] = frame[SEASON_COL].to_numpy()
frame_candidates["record_id"] = frame["record_id"].to_numpy()

display(frame_candidates[CANDIDATES].describe())
print("Medias por clase:")
display(frame_candidates.groupby(LABEL_COL)[CANDIDATES].mean())

## 5 — Ablación (puerta de promoción)

Con el mejor modelo base, bajo ambos protocolos: con vs sin
`cluster_meteo` + `score_anomalia`. `ablation_verdict` calcula ΔPR-AUC por
pliegue, media, **IC bootstrap** (semilla 42) y mínimo por protocolo. Se
promueve solo si la media supera el margen, el IC excluye 0 y ningún protocolo
queda negativo. Controles negativos: se añade `month`/`ccaa` one-hot como si
fuesen candidatos (no deben promoverse).

_[ENG] Ablation gate for the candidates under both protocols (mean delta PR-AUC, bootstrap CI, per-protocol min); month/ccaa one-hot additions act as negative controls._

In [ ]:
candidate_feature_cols = EVERYDAY_FEATURES + [c for c in CANDIDATES if c in frame_candidates.columns]
candidate_cat_cols = [
    c for c in categorical_columns(include_candidates=True) if c in candidate_feature_cols
]
best_model = model_candidates(seed=SEED)[best_name]

with_metrics = fold_eval(frame_candidates, candidate_feature_cols, candidate_cat_cols, best_model)
without_metrics = fold_eval(frame_candidates, EVERYDAY_FEATURES, categorical_columns(), best_model)
verdict = ablation_verdict(with_metrics, without_metrics, seed=SEED)
display(pd.DataFrame([verdict]))

# Controles negativos por el mismo veredicto.
frame_controls = frame_candidates.copy()
frame_controls["control_month"] = frame_controls["month"].astype(str)
frame_controls["control_ccaa"] = frame_controls["ccaa"].astype(str)
for control in ("control_month", "control_ccaa"):
    ctrl_with = fold_eval(
        frame_controls,
        EVERYDAY_FEATURES + [control],
        categorical_columns() + [control],
        best_model,
    )
    ctrl_verdict = ablation_verdict(ctrl_with, without_metrics, seed=SEED)
    print(
        f"Control {control}: promovido={ctrl_verdict['promoted']} "
        f"(delta medio {ctrl_verdict['mean_delta']:+.4f})"
    )

print(
    f"Candidatos promovidos: {verdict['promoted']} "
    f"(delta medio {verdict['mean_delta']:+.4f}, "
    f"IC95% [{verdict['ci_low']:+.4f}, {verdict['ci_high']:+.4f}])"
)

## 6 — Salidas

Predicciones out-of-fold (id, etiqueta, score, protocolo, grupo retenido) y
tabla de métricas por pliegue; curvas PR (OOF por protocolo), mapa de PR-AUC
por CCAA, importancia por permutación del modelo base y versiones de paquetes.
El texto de métodos cierra con los límites honestos.

_[ENG] Save out-of-fold predictions and the fold-metrics table; PR curves, per-CCAA choropleth, permutation importance and package versions; the methods note closes the notebook._

In [ ]:
def collect_oof(data, feature_cols, categorical_cols, model, protocol):
    """Scores out-of-fold para un protocolo (id, label, score, grupo)."""
    frames = []
    for split in blocked_splits(data, protocol):
        train, test = data.iloc[split.train_index], data.iloc[split.test_index]
        if train[LABEL_COL].nunique() < 2:
            continue
        X_train, X_test = prepare_fold(train, test, feature_cols, categorical_cols)
        fitted = clone(model).fit(X_train, train[LABEL_COL])
        out = test[["record_id", LABEL_COL]].copy()
        out["score"] = fitted.predict_proba(X_test)[:, 1]
        out["protocol"] = protocol
        out["held_out"] = split.held_out
        frames.append(out)
    return pd.concat(frames, ignore_index=True)


oof = pd.concat(
    [collect_oof(frame, EVERYDAY_FEATURES, categorical_columns(), floor_model, p) for p in PROTOCOLS],
    ignore_index=True,
)
save_csv(oof, PREDICTIONS_CSV)
save_csv(candidate_metrics, FOLD_METRICS_CSV)
print(f"Predicciones -> {PREDICTIONS_CSV} ({len(oof):,} filas)")
print(f"Metricas por pliegue -> {FOLD_METRICS_CSV}")

fig, axes = plt.subplots(1, len(PROTOCOLS), figsize=(5.5 * len(PROTOCOLS), 4))
for ax, protocol in zip(np.atleast_1d(axes), PROTOCOLS):
    subset = oof[oof["protocol"] == protocol]
    PrecisionRecallDisplay.from_predictions(subset[LABEL_COL], subset["score"], ax=ax)
    ax.set_title(f"PR (OOF) — bloqueo por {protocol}")
plt.tight_layout()
plt.show()

cca_metrics = candidate_metrics[
    (candidate_metrics["protocol"] == "ccaa") & (candidate_metrics["model"] == best_name)
]
boundaries = load_boundaries()
boundaries["ccaa"] = boundaries["name"].map(REGION_NAME_MAP).fillna(boundaries["name"])
map_data = boundaries.merge(
    cca_metrics[["held_out", "pr_auc"]], left_on="ccaa", right_on="held_out", how="left"
)
fig, ax = plt.subplots(figsize=(9, 7))
map_data.plot(column="pr_auc", cmap="YlOrRd", legend=True, missing_kwds={"color": "lightgrey"}, ax=ax)
ax.set_title(f"PR-AUC por CCAA (bloqueo por ccaa, {best_name})")
ax.axis("off")
plt.tight_layout()
plt.show()

first_split = blocked_splits(frame, "season")[0]
train, test = frame.iloc[first_split.train_index], frame.iloc[first_split.test_index]
X_train, X_test = prepare_fold(train, test, EVERYDAY_FEATURES, categorical_columns())
fitted = clone(floor_model).fit(X_train, train[LABEL_COL])
importance = permutation_importance(
    fitted, X_test, test[LABEL_COL], n_repeats=10, random_state=SEED, scoring="average_precision"
)
importance_table = pd.Series(importance.importances_mean, index=X_test.columns).sort_values(ascending=False)
ax = importance_table.head(20)[::-1].plot.barh(figsize=(8, 6))
ax.set_title("Importancia por permutacion (logistic, PR-AUC)")
ax.set_xlabel("Caida media de PR-AUC")
plt.tight_layout()
plt.show()

# Tasas empíricas CCAA×día (Plan D): referencia operativa del ranking.
fire_rates = ccaa_day_fire_rates(positives)
print("Tasas empíricas CCAA×día (Plan D):")
display(fire_rates)

print(
    f"numpy {np.__version__} | pandas {pd.__version__} | "
    f"scikit-learn {sklearn.__version__} | geopandas {geopandas.__version__} | "
    f"matplotlib {matplotlib.__version__}"
)

## Nota de métodos

- **Ranking, no probabilidad**: las puntuaciones ordenan; no son frecuencias
  calibradas. La prevalencia la fija el muestreo de fondo (ratio objetivo
  `background_ratio`; si el pool es menor, se usa entero), no la
  naturaleza. La PR-AUC de referencia es la prevalencia del pliegue.
- **Muestreo de fondo**: días de fondo por CCAA×día, con clima acotado por
  cuota de API (no todas las horas/lugares), y filtrado de fugas a
  ≤ `leakage_max_km` de una detección del mismo día.
- **Validación bloqueada**: leave-one-group-out por temporada y CCAA; el
  one-hot y el escalador se ajustan solo en entrenamiento.
- **Reintegración**: `cluster_meteo`/`score_anomalia` se ajustan solo con
  positivos; se promueven solo si la ablación lo justifica.
- **Procedencia del PCA**: verificada 2026-10-18 — `fit_weather_pca`
  reproduce el espacio de `features_a.csv`/03b (PC1–7, k=7, varianza
  acumulada 0.9152; `tests/test_pca_weather.py::TestAgainstRealFeaturesA`).
- **Tasas empíricas CCAA×día**: se reportan (fila `ALL` + por CCAA) como
  referencia operativa; la calibración del score contra ellas sigue
  [ABIERTA] (Plan D), y el modelo se mantiene en modo *ranking*.